In [ ]:
import os
import pandas as pd
import numpy as np
import xgboost as xgb
import bottleneck as bn

from utils import download_crypto_data
from datetime import date
from sklearn.metrics import classification_report, roc_auc_score, accuracy_score, precision_recall_curve

In [ ]:
download_crypto_data(symbol="BTCUSDT", start=date(2020, 1, 1), end=date(2024, 12, 31), interval="5m", suppress_info=True)

In [ ]:
train_data = pd.read_csv("data/BTCUSDT-5m-2020-01-01_2024-12-31.csv")
test_data = pd.read_csv("data/BTCUSDT-5m-2025-01-01_2026-06-30.csv")
train_df = pd.DataFrame()
test_df = pd.DataFrame()

Feature Engineering

In [ ]:
t_cost = (0.1 + 0.02 + 0.015) * 0.01  # Transaction cost by Binance + slippage + liquidity, or set to zero if wanted

train_data["RelativeDiff"] = (train_data["High"] - train_data["Low"]) / (train_data["High"] + train_data["Low"])
test_data["RelativeDiff"] = (test_data["High"] - test_data["Low"]) / (test_data["High"] + test_data["Low"])

windows = [10, 25, 50]
configs = ["Volume", "NumberOfTrades", "Close"]

for source_col in configs:
    train_rolling = {w: train_data[source_col].rolling(w) for w in windows}
    test_rolling = {w: test_data[source_col].rolling(w) for w in windows}

    for w in windows:
        train_df[f"FEATURE_{source_col}Zscore{w}"] = (train_data[source_col] - train_rolling[w].mean()) / train_rolling[w].std()
        test_df[f"FEATURE_{source_col}Zscore{w}"] = (test_data[source_col] - test_rolling[w].mean()) / test_rolling[w].std()

ema_windows = [10, 20, 50]
for w in ema_windows:
    train_df[f"FEATURE_RelativeEMA{w}"] = train_data["Close"].ewm(span=w, min_periods=w, adjust=False).mean() / train_data["Close"]
    test_df[f"FEATURE_RelativeEMA{w}"] = test_data["Close"].ewm(span=w, min_periods=w, adjust=False).mean() / test_data["Close"]

macd_settings = [(10, 20), (10, 50), (20, 50)]
macd_frames_train = [train_df]
macd_frames_test = [test_df]
for fast, slow in macd_settings:
    macd_df_train = train_data.ta.macd(close='Close', fast=fast, slow=slow, signal=9).iloc[:, 0].to_frame()
    macd_df_train = macd_df_train.add_suffix(f'_f{fast}_s{slow}').add_prefix("FEATURE_")
    macd_frames_train.append(macd_df_train)
    macd_df_test = test_data.ta.macd(close='Close', fast=fast, slow=slow, signal=9).iloc[:, 0].to_frame()
    macd_df_test = macd_df_test.add_suffix(f'_f{fast}_s{slow}').add_prefix("FEATURE_")
    macd_frames_test.append(macd_df_test)
train_df = pd.concat(macd_frames_train, axis=1)
test_df = pd.concat(macd_frames_test, axis=1)

rolling_TrainRelativeDiffPct50 = bn.move_rank(train_data["RelativeDiff"].values, window=50)
rolling_TestRelativeDiffPct50 = bn.move_rank(test_data["RelativeDiff"].values, window=50)

train_df[f"FEATURE_RelativeDiffPct50"] = rolling_TrainRelativeDiffPct50 / 50.0
test_df[f"FEATURE_RelativeDiffPct50"] = rolling_TestRelativeDiffPct50 / 50.0

train_df[f"TARGET_UpFlag"] = (train_data["Open"] * (1 + t_cost) < train_data["Close"]).shift(-1)
test_df[f"TARGET_UpFlag"] = (test_data["Open"] * (1 + t_cost) < test_data["Close"]).shift(-1)

train_df = train_df.dropna()
test_df = test_df.dropna()
train_df["TARGET_UpFlag"] = train_df["TARGET_UpFlag"].astype(int)
test_df["TARGET_UpFlag"] = test_df["TARGET_UpFlag"].astype(int)

XGBoost Training

In [ ]:
# 1. Separate features (X) and target (y)
target_col = "TARGET_UpFlag"
random_state = 60  # Used when fitting XGB Classifier

X_train = train_df.drop(columns=[target_col])
y_train = train_df[target_col]

X_test = test_df.drop(columns=[target_col])
y_test = test_df[target_col]

num_neg = (y_train == 0).sum()
num_pos = (y_train == 1).sum()
dampened_neg_pos_ratio = (1 + num_neg / num_pos) / 2
# 2. Initialize XGBoost Classifier
model = xgb.XGBClassifier(
    n_estimators=3000,          # Max number of trees
    max_depth=3,                # Decision rules depth
    learning_rate=0.016,
    subsample=0.7,
    colsample_bytree=0.3,
    min_child_weight=120,        # Requires at least 80 samples to create a terminal leaf node
    gamma=5.0,
    random_state=random_state,
    eval_metric='auc',
    n_jobs=-1,
    early_stopping_rounds=150,   # Stops fitting when no improvement after consecutive 150 rounds
    scale_pos_weight=dampened_neg_pos_ratio,
    reg_alpha=3.0,               # L1 regularization to naturally prune useless features
    reg_lambda=30.0              # L2 regularization to smooth out extreme weight fluctuations
)

# 3. Train & save the model
# We use the test set as validation to monitor early stopping
print("Training XGBoost model...")
model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=40,                  # Print progress every 40 trees
)

os.makedirs("models", exist_ok=True)
model.save_model("models/xgboost_1h_kline.json")

# 4. Evaluate the model
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

precisions, recalls, thresholds = precision_recall_curve(y_test, y_prob)

f1_scores = (2 * precisions * recalls) / (precisions + recalls + 1e-10)

best_idx = np.argmax(f1_scores[: -1])
best_threshold = thresholds[best_idx]

y_pred_custom = (y_prob >= best_threshold).astype(int)

print(f"\nDynamically Chosen Best Threshold (Max F1): {best_threshold:.4f}")
print(f"Expected Max F1-Score: {f1_scores[best_idx]:.4f}")
print("=== Evaluation Results ===")
print(f"Accuracy: {accuracy_score(y_test, y_pred_custom):.4f}")
print(f"ROC-AUC Score: {roc_auc_score(y_test, y_prob):.4f}")
print(f"\nClassification Report [Threshold={best_threshold:.4f}]:")
print(classification_report(y_test, y_pred_custom))

# 5. Check Feature Importance
importance = pd.Series(model.feature_importances_, index=X_train.columns)
print("\n=== Features Importance ===")
print(importance.nlargest(50))